# RevPlan Simulation — MLWB run (real pipeline)
Reads the params handed over by the Action Flow, then runs the **real** pipeline
via the engine's own `read_inputs` → `run_simulation` → `write_outputs`, and
resumes the orchestration when finished.

This reads **real Celonis data** (not synthetic): tables with a source are pulled
from the Data Model; tables that have no source yet return `celonis_io`'s built-in
placeholders (empty frames). So it needs the Data Model + pool/table names in
`celonis_io` to resolve — and, until the PRODID↔MODEL_NO crosswalk and
`equipment_to_process` are supplied, the allocation output may be sparse.

In [ ]:
# papermill overrides these from the Action Flow /executions params (cell tagged 'parameters').
# The values here are the defaults used when the notebook is run manually.
#
# IMPORTANT — dpInstanceId is the key the OE correlates on to resume the waiting
# 'finish-simulation' step (sent as the ce-instanceid header, see cell 8). The
# Action Flow MUST pass the real Digital Process Instance ID here, i.e. map
# "dpInstanceId": "{{1.instanceId}}" in the trigger's /executions params JSON.
# If it is left as the 'local-test' sentinel below, the final cell SKIPS the resume
# event rather than notifying the OE with a bogus id it can never match.
dpInstanceId = 'local-test'
scenario_name = 'Baseline Test 0616'
revenue_plan_id = '1'
weight_revenue = '50'
weight_margin = '30'
weight_delivery = '20'
start_month = '202602'
max_delay_days = '200'
prototype_pct = '0'

In [ ]:
# Ensure runtime deps exist in THIS kernel (installs only if missing). papermill runs
# this cell too, so triggered executions self-heal even if a workbench restart wiped them.
# Uses sys.executable so it always targets the notebook's own env (not the terminal's).
# oauthlib / requests_oauthlib back the OE client-credentials token (orchestration.py).
import importlib.util, subprocess, sys
for _pkg, _mod in (('polars', 'polars'), ('pycelonis', 'pycelonis'), ('requests', 'requests'),
                   ('oauthlib', 'oauthlib'), ('requests_oauthlib', 'requests_oauthlib')):
    if importlib.util.find_spec(_mod) is None:
        print('installing', _pkg, '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)
print('deps present; python =', sys.executable)

In [ ]:
# ── OE credentials ──────────────────────────────────────────────────────────────
# Precedence: real environment variable  >  hardcoded block below  >  .env file.
import os
from pathlib import Path

# ⚠️  HARDCODED STOPGAP — OE OAuth client "OE ConnectionTest", PLAINTEXT SECRET.
#     This gets the run working without a .env / MLWB env var. Before you SHARE or
#     COMMIT this notebook: ROTATE this secret and move it into a .env or an MLWB
#     environment variable, then delete this block. setdefault() means a real env
#     var already set for these names wins over these literals.
os.environ.setdefault('oe_client_id',     '44d222ca-6e8a-4258-8216-81672c292645')
os.environ.setdefault('oe_client_secret', '-0nOcqUqm2jTsIDU3DCPUsXmpB3rZx4pgqsyuvLfQ8_yQbu77qF03vjLz9tlGROx')
# OE package KEY (slug with underscores) — NOT the package id (hyphenated UUID in the
# Studio URL). This is what /oe/api/packages/{key}/messages resolves against.
os.environ.setdefault('OE_PACKAGE_KEY',   '36f39662_626e_41c2_be62_fcd0ead2b2ce')

# Optional: also load a .env (next to this notebook or a parent dir) — stdlib only,
# never overrides an already-set env var. Lets you swap to a .env later with no code change.
def _load_dotenv(path):
    """KEY=VALUE per line; skips comments/blank; never overrides an existing env var.
    Returns the list of key NAMES loaded (never values)."""
    if not path.exists():
        return []
    loaded = []
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        if line.startswith('export '):
            line = line[len('export '):]
        key, _, val = line.partition('=')
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = val
            loaded.append(key)
    return loaded

# Search cwd + a few ancestors + home for a .env (MLWB cwd under papermill is unpredictable).
_cands, _d, _seen = [], os.getcwd(), set()
for _ in range(6):
    _cands.append(Path(_d) / '.env'); _d = os.path.dirname(_d)
_cands.append(Path(os.path.expanduser('~')) / '.env')
for _p in _cands:
    if str(_p) in _seen:
        continue
    _seen.add(str(_p))
    _names = _load_dotenv(_p)
    if _names:
        print(f"[env] loaded {_names} from {_p}")

# Report presence WITHOUT printing secret values.
_id_ok  = bool(os.getenv('oe_client_id')  or os.getenv('OE_CLIENT_ID'))
_sec_ok = bool(os.getenv('oe_client_secret') or os.getenv('OE_CLIENT_SECRET'))
print(f"[env] oe_client_id: {'present' if _id_ok else 'MISSING'} | "
      f"oe_client_secret: {'present' if _sec_ok else 'MISSING'} | "
      f"OE_PACKAGE_KEY: {os.getenv('OE_PACKAGE_KEY')}")

In [ ]:
import os, sys
# This notebook may sit INSIDE revplan_engine/ or beside it, and papermill's cwd is
# unpredictable. Find the folder that CONTAINS 'revplan_engine' and put THAT on sys.path.
def _find_engine_parent():
    cands = []
    env = os.environ.get('ENGINE_PARENT')
    if env:
        cands.append(env)
    d = os.getcwd()
    # case: cwd IS the package dir (notebook uploaded inside revplan_engine) -> go one up
    if os.path.basename(d) == 'revplan_engine' and os.path.isfile(os.path.join(d, 'run_simulation.py')):
        cands.append(os.path.dirname(d))
    for _ in range(6):                       # cwd + ancestors
        cands.append(d); d = os.path.dirname(d)
    cands += [os.path.expanduser('~'), '/home/jovyan']
    for base in cands:
        if base and os.path.isfile(os.path.join(base, 'revplan_engine', 'run_simulation.py')):
            return base
    home = os.path.expanduser('~')           # last resort: bounded search under home
    for root, dirs, _ in os.walk(home):
        if root[len(home):].count(os.sep) > 4:
            dirs[:] = []; continue
        dirs[:] = [x for x in dirs if not x.startswith('.') and x not in ('node_modules', '__pycache__', 'site-packages')]
        if os.path.isfile(os.path.join(root, 'revplan_engine', 'run_simulation.py')):
            return root
    return None

print('cwd =', os.getcwd())
parent = _find_engine_parent()
if parent is None:
    raise ModuleNotFoundError(
        "revplan_engine not found. Move this notebook one level ABOVE the revplan_engine "
        "folder (so it sits beside it), or set ENGINE_PARENT. Locate with: "
        "!find ~ -maxdepth 5 -name run_simulation.py")
if parent not in sys.path:
    sys.path.insert(0, parent)
from revplan_engine.run_simulation import SimulationParams, run_simulation
print('imports OK; engine parent =', parent)

In [ ]:
# --- Resolve the real Digital Process Instance ID -------------------------------
# The Action Flow should inject it as 'dpInstanceId'. Accept a few common alternate
# spellings too, so a naming/casing mismatch in the /executions params doesn't
# silently strand the OE. papermill injects its overrides in a cell ABOVE this one,
# so any alternate the flow sent is already a global by now.
PLACEHOLDER_INSTANCE_IDS = {'', 'local-test', 'none', 'None', 'null'}
for _alt in ('dpInstanceId', 'instanceId', 'instance_id', 'dp_instance_id'):
    _v = globals().get(_alt)
    if _v is not None and str(_v).strip() and str(_v).strip() not in PLACEHOLDER_INSTANCE_IDS:
        dpInstanceId = str(_v).strip()
        break
dpInstanceId_is_real = str(dpInstanceId).strip() not in PLACEHOLDER_INSTANCE_IDS

received = {
    'dpInstanceId': dpInstanceId, 'scenario_name': scenario_name,
    'revenue_plan_id': revenue_plan_id, 'weight_revenue': weight_revenue,
    'weight_margin': weight_margin, 'weight_delivery': weight_delivery,
    'start_month': start_month, 'max_delay_days': max_delay_days,
    'prototype_pct': prototype_pct,
}
print('=== PARAMS RECEIVED FROM ACTION FLOW ===')
for k, v in received.items():
    print('  ', k, '=', repr(v))
if not dpInstanceId_is_real:
    print("   ⚠️  dpInstanceId is a PLACEHOLDER — the OE resume callback will be SKIPPED "
          "at the end of this run.\n"
          "       Pass {{1.instanceId}} as 'dpInstanceId' in the Action Flow /executions params "
          "to close the orchestration loop.")

_wr, _wm, _wd = float(weight_revenue), float(weight_margin), float(weight_delivery)
# Weighted priority is EXPLICIT OPT-IN ONLY (2026-07-15 meeting: weights judged not
# practically useful; 'target_step' is the customer's default rule). The Action Flow
# always sends default weights, so inferring intent from non-zero weights silently
# forced weighted mode on every run. Weights are kept on the params for a deliberate
# opt-in via a 'use_weighted_priority' papermill param.
_use_weighted = str(globals().get('use_weighted_priority', '')).strip().lower() in ('1', 'true', 'yes', 'y')
if not _use_weighted and (_wr + _wm + _wd) > 0:
    print(f"   \u2139 weights ({_wr}, {_wm}, {_wd}) received but weighted priority stays OFF "
          "(2026-07-15 decision; pass use_weighted_priority=true to opt in)")
params = SimulationParams(
    simulation_id=dpInstanceId, simulation_name=scenario_name,
    revenue_plan_id=revenue_plan_id, start_month=str(start_month),
    max_delay_days=int(max_delay_days), weight_revenue=_wr,
    weight_margin=_wm, weight_delivery=_wd,
    prototype_pct=float(prototype_pct), use_weighted_priority=_use_weighted,
)
print('built SimulationParams for', params.simulation_name,
      '| use_weighted_priority (intent) =', params.use_weighted_priority)

In [ ]:
# Real Celonis reads via the engine's own I/O layer. Tables with a source are pulled
# from the Data Model; tables with no source yet return celonis_io's built-in
# placeholders (empty frames) — i.e. the "leave as placeholders" behaviour, but the
# rest is the real pipeline reading real data.
from revplan_engine.celonis_io import read_inputs
inputs = read_inputs(params)
print('=== read_inputs (row counts) ===')
for name, df in inputs.items():
    print('  {:24s} {:>8} rows'.format(name, df.height))

In [ ]:
results = run_simulation(params, inputs=inputs)
print('=== RESULT TABLES (row counts) ===')
for name, df in results.items():
    print('  {:32s} {:>6} rows'.format(name, df.height))

In [ ]:
# Push the SIM_* result tables back to the Data Pool. Best-effort: a pool/permission
# issue is caught and printed so it doesn't kill the run or the resume callback.
WRITE_OUTPUTS = True
if WRITE_OUTPUTS:
    try:
        from revplan_engine.celonis_io import write_outputs
        write_outputs(results, params)
        print('write_outputs: done')
    except Exception as ex:
        print('write_outputs skipped/failed:', ex)
else:
    print('write_outputs disabled')

In [ ]:
# ---- Resume the OE: emit the 'finish-simulation' completion event ----
# /executions is fire-and-forget, so THIS event is what advances the orchestration
# past its 'finish-simulation' resume step and closes the loop. Uses the OE's native
# message API (POST /oe/api/packages/{OE_PACKAGE_KEY}/messages) with the CloudEvents
# ce-instanceid (=dpInstanceId) + ce-type (=finish-simulation) headers — see
# revplan_engine/orchestration.py. Config comes from the environment / .env:
#   OE_PACKAGE_KEY, oe_client_id, oe_client_secret  (OE_EVENT_TYPE defaults to finish-simulation)
from revplan_engine.orchestration import emit_finished_simulation

if not dpInstanceId_is_real:
    # Manual/dev run, or the Action Flow did not pass the instance id. Do NOT notify
    # the OE with the placeholder: a bogus id can't be matched to the waiting instance
    # and would look like a phantom 'finished' event. (This silent-bogus-notify was the
    # original bug — the run reported success while the OE never advanced.)
    print(f"SKIP OE notify: dpInstanceId is a placeholder ({dpInstanceId!r}); "
          "nothing sent to the OE.")
else:
    # emit_finished_simulation RAISES on a non-2xx (a failed resume strands the OE,
    # so it must be loud). event_type defaults to OE_EVENT_TYPE / 'finish-simulation'.
    emit_finished_simulation(
        dpInstanceId,
        body={'simulation_id': dpInstanceId, 'scenario_name': scenario_name, 'status': 'COMPLETE'},
        event_type=os.environ.get('OE_EVENT_TYPE', 'finish-simulation'),
    )